# TIMER-Graph: Research Visualizations\n\n**Purpose**: Generate publication-quality figures for TIMER-Graph research paper.\n\n**Outputs**:\n1. Phase Progression Chart (Recall@5 across phases)\n2. Hard Negative Performance Comparison\n3. Temporal Decay Curves\n4. Beta Modulation Visualization

In [ ]:
# Setup\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport json\nfrom pathlib import Path\n\n# Publication style\nplt.style.use('seaborn-v0_8-whitegrid')\nplt.rcParams['figure.dpi'] = 300\nplt.rcParams['font.size'] = 10\nplt.rcParams['axes.titlesize'] = 12\nplt.rcParams['axes.labelsize'] = 10\n\n# Paths\nROOT = Path('..').resolve()\nDATA_EVAL = ROOT / 'data' / 'evaluation'\nRESULTS_P5 = ROOT / 'results' / 'phase5'\nPLOTS_DIR = ROOT / 'visualizations' / 'plots' / 'publication_ready'\nPLOTS_DIR.mkdir(parents=True, exist_ok=True)\n\nprint(f'Root: {ROOT}')\nprint(f'Plots will be saved to: {PLOTS_DIR}')

## 1. Phase Progression Chart\nShows Recall@5 improvement across all research phases.

In [ ]:
# Load Phase Results\nwith open(DATA_EVAL / 'phase_1_baseline_results.json') as f:\n    p1 = json.load(f)\nwith open(DATA_EVAL / 'phase_3_results.json') as f:\n    p3 = json.load(f)\n\n# Phase 5 Results\np5_summary = pd.read_csv(RESULTS_P5 / 'summary_table.csv')\ntimer_accuracy = float(p5_summary[p5_summary['Scenario'] == '**OVERALL**']['TIMER Accuracy'].values[0].replace('%', '')) / 100\n\n# Compile data\nphases = ['Naive\\n(Baseline)', 'Hybrid\\nChunking', 'Cross-Encoder\\nReranking', 'TIMER-Graph\\n(Hard Neg)']\nrecall_values = [\n    p1['results']['Naive']['Recall@5'],\n    p1['results']['POC (Hybrid)']['Recall@5'],\n    p3['results']['Phase 3 (Reranking)']['Recall@5'],\n    timer_accuracy\n]\n\nprint('Phase Progression Data:')\nfor p, r in zip(phases, recall_values):\n    print(f'  {p.replace(chr(92)+chr(110), " ")}: {r:.2%}')

In [ ]:
# Figure 1: Phase Progression\nfig, ax = plt.subplots(figsize=(8, 5))\n\ncolors = ['#E57373', '#FFB74D', '#64B5F6', '#81C784']\nbars = ax.bar(phases, recall_values, color=colors, edgecolor='black', linewidth=0.8)\n\n# Add value labels\nfor bar, val in zip(bars, recall_values):\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, \n            f'{val:.0%}', ha='center', va='bottom', fontsize=11, fontweight='bold')\n\nax.set_ylabel('Recall@5', fontsize=12)\nax.set_title('Research Journey: Retrieval Performance Across Phases', fontsize=14, fontweight='bold')\nax.set_ylim(0, 1.1)\nax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random Baseline')\n\n# Improvement arrows\nax.annotate('', xy=(3, timer_accuracy), xytext=(0, recall_values[0]),\n            arrowprops=dict(arrowstyle='->', color='green', lw=2, ls='--'))\nax.text(1.5, 0.75, '+37%', fontsize=12, color='green', fontweight='bold')\n\nplt.tight_layout()\nplt.savefig(PLOTS_DIR / 'fig1_phase_progression.pdf', bbox_inches='tight')\nplt.savefig(PLOTS_DIR / 'fig1_phase_progression.png', bbox_inches='tight')\nprint(f'Saved: {PLOTS_DIR / "fig1_phase_progression.pdf"}')

## 2. Hard Negative Performance (Main Figure)\nCompares Baseline vs TIMER accuracy on hard negative scenarios.

In [ ]:
# Load Phase 5 Summary\ndf = pd.read_csv(RESULTS_P5 / 'summary_table.csv')\ndf = df[df['Scenario'] != '**OVERALL**']  # Exclude overall for per-scenario\n\n# Parse percentages\ndf['Baseline'] = df['Baseline Accuracy'].str.replace('%', '').astype(float)\ndf['TIMER'] = df['TIMER Accuracy'].str.replace('%', '').astype(float)\n\nprint(df[['Scenario', 'Baseline', 'TIMER']])

In [ ]:
# Figure 2: Hard Negative Comparison\nfig, ax = plt.subplots(figsize=(10, 6))\n\nx = np.arange(len(df))\nwidth = 0.35\n\nbars1 = ax.bar(x - width/2, df['Baseline'], width, label='Semantic Baseline', color='#BDBDBD', edgecolor='black')\nbars2 = ax.bar(x + width/2, df['TIMER'], width, label='TIMER-Graph', color='#2196F3', edgecolor='black')\n\n# Labels\nax.set_ylabel('Accuracy (%)', fontsize=12)\nax.set_title('Hard Negative Stress Test: TIMER-Graph vs Semantic Baseline', fontsize=14, fontweight='bold')\nax.set_xticks(x)\nax.set_xticklabels(df['Scenario'], rotation=15, ha='right')\nax.legend(loc='upper left')\nax.set_ylim(0, 120)\n\n# Value labels\nfor bar in bars1:\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,\n            f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9)\nfor bar in bars2:\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,\n            f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')\n\n# Highlight improvement\nax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Random Chance')\n\nplt.tight_layout()\nplt.savefig(PLOTS_DIR / 'fig2_hard_negative_comparison.pdf', bbox_inches='tight')\nplt.savefig(PLOTS_DIR / 'fig2_hard_negative_comparison.png', bbox_inches='tight')\nprint(f'Saved: {PLOTS_DIR / "fig2_hard_negative_comparison.pdf"}')

## 3. Temporal Decay Curves\nVisualizes the exponential decay function with different lambda values.

In [ ]:
# Figure 3: Decay Curves\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n\n# 3a: Basic decay\nt = np.linspace(0, 2000, 500)\nlambdas = [0.001, 0.005, 0.01]\ncolors = ['#4CAF50', '#2196F3', '#F44336']\n\nfor lam, c in zip(lambdas, colors):\n    decay = np.exp(-lam * t)\n    ax1.plot(t, decay, label=f'λ = {lam}', color=c, linewidth=2)\n\nax1.set_xlabel('Time Offset (days)')\nax1.set_ylabel('Decay Value e^(-λt)')\nax1.set_title('(A) Exponential Decay Function', fontweight='bold')\nax1.legend()\nax1.set_xlim(0, 2000)\nax1.set_ylim(0, 1.05)\n\n# 3b: Beta modulation (with λ = 0.005)\nlam = 0.005\nsemantic = 0.6  # Fixed semantic score\nbetas = [0.8, 0.0, -0.3]\nbeta_labels = ['β=+0.8 (Current)', 'β=0.0 (Neutral)', 'β=-0.3 (Historical)']\ncolors = ['#4CAF50', '#9E9E9E', '#F44336']\n\nfor beta, label, c in zip(betas, beta_labels, colors):\n    score = semantic + beta * np.exp(-lam * t)\n    ax2.plot(t, score, label=label, color=c, linewidth=2)\n\nax2.axhline(y=semantic, color='black', linestyle=':', alpha=0.5, label='Semantic Only')\nax2.set_xlabel('Time Offset (days)')\nax2.set_ylabel('Final TIMER Score')\nax2.set_title('(B) Intent-Modulated Scoring', fontweight='bold')\nax2.legend(loc='center right')\nax2.set_xlim(0, 1000)\nax2.set_ylim(0.2, 1.5)\n\n# Annotate flip zone\nax2.fill_between(t[(t > 0) & (t < 200)], 0.2, 1.5, alpha=0.1, color='orange')\nax2.text(100, 1.35, 'Recent\\nActive Zone', ha='center', fontsize=9, color='orange')\n\nplt.tight_layout()\nplt.savefig(PLOTS_DIR / 'fig3_temporal_decay.pdf', bbox_inches='tight')\nplt.savefig(PLOTS_DIR / 'fig3_temporal_decay.png', bbox_inches='tight')\nprint(f'Saved: {PLOTS_DIR / "fig3_temporal_decay.pdf"}')

## 4. Intent Modulation: Beta Values\nShows the beta parameter values for each intent type.

In [ ]:
# Figure 4: Beta Values Bar Chart\nfig, ax = plt.subplots(figsize=(6, 4))\n\nintents = ['Current', 'Historical', 'Trend']\nbetas = [0.8, -0.3, 0.4]  # From TIMERScorer\ncolors = ['#4CAF50', '#F44336', '#2196F3']\n\nbars = ax.bar(intents, betas, color=colors, edgecolor='black', linewidth=1)\n\n# Zero line\nax.axhline(y=0, color='black', linewidth=1)\n\n# Value labels\nfor bar, val in zip(bars, betas):\n    offset = 0.05 if val > 0 else -0.1\n    ax.text(bar.get_x() + bar.get_width()/2, val + offset,\n            f'{val:+.1f}', ha='center', va='bottom' if val > 0 else 'top',\n            fontsize=12, fontweight='bold')\n\nax.set_ylabel('β (Beta) Value', fontsize=12)\nax.set_title('Intent-Modulated Beta Parameters', fontsize=14, fontweight='bold')\nax.set_ylim(-0.6, 1.0)\n\n# Annotations\nax.text(0, 0.95, 'Boost Recent', ha='center', fontsize=9, style='italic', color='#388E3C')\nax.text(1, -0.55, 'Boost Historical', ha='center', fontsize=9, style='italic', color='#D32F2F')\n\nplt.tight_layout()\nplt.savefig(PLOTS_DIR / 'fig4_beta_values.pdf', bbox_inches='tight')\nplt.savefig(PLOTS_DIR / 'fig4_beta_values.png', bbox_inches='tight')\nprint(f'Saved: {PLOTS_DIR / "fig4_beta_values.pdf"}')

## 5. Ranking Flip Case Study\nDemonstrates how TIMER reverses ranking for a historical query.

In [ ]:
# Load specific case study from semantic collision\nsc_results = pd.read_csv(RESULTS_P5 / 'semantic_collision_results.csv')\ncase = sc_results[sc_results['query_id'] == 'sc_hist_1'].iloc[0]\n\nprint('Case Study: sc_hist_1')\nprint(f'Query: {case["query_text"]}')\nprint(f'Expected: {case["expected_note"]}')\nprint(f'Baseline Retrieved: {case["baseline_retrieval"]} (Correct: {case["baseline_correct"]})')\nprint(f'TIMER Retrieved: {case["timer_retrieval"]} (Correct: {case["timer_correct"]})')

In [ ]:
# Figure 5: Ranking Flip Visualization\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))\n\n# Simulated scores (based on formula)\nsemantic_score = 0.6  # Identical for both\noffset_new, offset_old = 10, 500\nlam = 0.005\nbeta_hist = -0.3\n\n# Baseline (semantic only)\nnotes_baseline = ['New Note\\n(10 days)', 'Old Note\\n(500 days)']\nscores_baseline = [semantic_score, semantic_score]\n\n# TIMER with historical intent\nscore_new = semantic_score + beta_hist * np.exp(-lam * offset_new)\nscore_old = semantic_score + beta_hist * np.exp(-lam * offset_old)\nscores_timer = [score_new, score_old]\n\n# Plot Baseline\nbars1 = ax1.barh(notes_baseline, scores_baseline, color=['#BDBDBD', '#BDBDBD'], edgecolor='black')\nax1.set_xlim(0, 1.0)\nax1.set_xlabel('Score')\nax1.set_title('(A) Semantic Baseline\\n(New Note Wins ❌)', fontweight='bold', color='red')\nax1.axvline(x=0.6, color='gray', linestyle='--', alpha=0.5)\nfor bar, val in zip(bars1, scores_baseline):\n    ax1.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')\n\n# Plot TIMER\ncolors_timer = ['#FFCDD2', '#C8E6C9']  # Light red, light green\nbars2 = ax2.barh(notes_baseline, scores_timer, color=colors_timer, edgecolor='black')\nax2.set_xlim(0, 1.0)\nax2.set_xlabel('Score')\nax2.set_title('(B) TIMER-Graph (Historical)\\n(Old Note Wins ✓)', fontweight='bold', color='green')\nax2.axvline(x=0.6, color='gray', linestyle='--', alpha=0.5)\nfor bar, val in zip(bars2, scores_timer):\n    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')\n\nplt.suptitle(f'Query: \"{case["query_text"]}\"', fontsize=11, style='italic', y=1.02)\nplt.tight_layout()\nplt.savefig(PLOTS_DIR / 'fig5_ranking_flip.pdf', bbox_inches='tight')\nplt.savefig(PLOTS_DIR / 'fig5_ranking_flip.png', bbox_inches='tight')\nprint(f'Saved: {PLOTS_DIR / "fig5_ranking_flip.pdf"}')

## Summary\n\nGenerated figures:\n1. `fig1_phase_progression.pdf` - Research journey chart\n2. `fig2_hard_negative_comparison.pdf` - Main performance figure\n3. `fig3_temporal_decay.pdf` - Decay mechanism visualization\n4. `fig4_beta_values.pdf` - Intent modulation parameters\n5. `fig5_ranking_flip.pdf` - Case study before/after

In [ ]:
# List generated files\nprint('\\n=== Generated Publication Figures ===')\nfor f in sorted(PLOTS_DIR.glob('*.pdf')):\n    print(f'  ✓ {f.name}')